<a href="https://colab.research.google.com/github/kruthee05/MACHINELEARNING2/blob/main/FirstOrder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import zipfile

# ==========================================
# 1. LOAD DATASET FROM ZIP FILE
# ==========================================

file_path = "/content/dataset_2191_sleep.csv"


df = pd.read_csv(file_path)

print("Dataset:")
print(df.head())

print("\nDataset Information:")
print(df.info())


# ==========================================
# 2. HANDLE MISSING VALUES
# ==========================================

# Replace '?' with NaN
df.replace("?", np.nan, inplace=True)

# Convert all columns to numeric
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Fill missing values using median
df.fillna(df.median(), inplace=True)


# ==========================================
# 3. CONVERT TOTAL_SLEEP INTO CLASSES
# ==========================================

def classify_sleep(hours):

    if hours <= 8:
        return "Low_Sleep"

    elif hours <= 12:
        return "Medium_Sleep"

    else:
        return "High_Sleep"


df["sleep_class"] = df["total_sleep"].apply(classify_sleep)

print("\nClass Distribution:")
print(df["sleep_class"].value_counts())


# ==========================================
# 4. DISCRETIZE NUMERICAL FEATURES
# FOIL WORKS BETTER WITH SYMBOLIC CONDITIONS
# ==========================================

features = [
    "body_weight",
    "brain_weight",
    "max_life_span",
    "gestation_time",
    "predation_index",
    "sleep_exposure_index",
    "danger_index"
]

# Create categorical versions of numerical columns
for feature in features:
    num_unique_values = df[feature].nunique()

    # Determine how many bins can actually be created meaningfully
    if num_unique_values <= 1:
        df[feature + "_cat"] = "Single_Category" # Assign a single category if too few unique values
        continue
    elif num_unique_values == 2:
        q_to_use = 2
        labels_to_use = ["Low", "High"]
    else: # num_unique_values >= 3
        q_to_use = 3
        labels_to_use = ["Low", "Medium", "High"]

    try:
        df[feature + "_cat"] = pd.qcut(
            df[feature],
            q=q_to_use,
            labels=labels_to_use,
            duplicates="drop"
        )
    except Exception as e:
        # Fallback for extreme cases where qcut might still fail
        print(f"Warning: Fallback applied for feature '{feature}' due to error: {e}")
        df[feature + "_cat"] = "Failed_Discretization" # Assign a neutral category


# ==========================================
# 5. SIMPLE FIRST ORDER INDUCTIVE LEARNER
# ==========================================

def calculate_accuracy(condition, target_class):

    matching = df.query(condition)

    if len(matching) == 0:
        return 0

    correct = matching[
        matching["sleep_class"] == target_class
    ]

    return len(correct) / len(matching)


# ==========================================
# 6. GENERATE CLASSIFICATION RULES
# ==========================================

rules = []

categorical_features = [
    feature + "_cat"
    for feature in features
]


for target_class in df["sleep_class"].unique():

    best_rule = None
    best_accuracy = 0
    best_support = 0

    # Try each feature-value combination
    for feature in categorical_features:

        for value in df[feature].dropna().unique():

            subset = df[df[feature] == value]

            if len(subset) == 0:
                continue

            correct = subset[
                subset["sleep_class"] == target_class
            ]

            accuracy = len(correct) / len(subset)

            support = len(correct)

            if accuracy > best_accuracy:

                best_accuracy = accuracy
                best_rule = (
                    f"IF {feature} = {value} "
                    f"THEN sleep_class = {target_class}"
                )
                best_support = support


    rules.append({
        "Class": target_class,
        "Rule": best_rule,
        "Accuracy": round(best_accuracy * 100, 2),
        "Correct Examples": best_support
    })


# ==========================================
# 7. DISPLAY LEARNED RULES
# ==========================================

print("\n" + "=" * 60)
print("FIRST ORDER INDUCTIVE LEARNER - CLASSIFICATION RULES")
print("=" * 60)

for rule in rules:

    print("\nTarget Class:", rule["Class"])
    print("Rule:", rule["Rule"])
    print("Accuracy:", rule["Accuracy"], "%")
    print("Correctly Covered Examples:", rule["Correct Examples"])


# ==========================================
# 8. DISPLAY FINAL DATASET
# ==========================================

print("\nFinal Dataset:")
print(df.head(10))

Dataset:
   body_weight  brain_weight max_life_span gestation_time  predation_index  \
0     6654.000        5712.0          38.6            645                3   
1        1.000           6.6           4.5             42                3   
2        3.385          44.5            14             60                1   
3        0.920           5.7             ?             25                5   
4     2547.000        4603.0            69            624                3   

   sleep_exposure_index  danger_index total_sleep  
0                     5             3         3.3  
1                     1             3         8.3  
2                     1             1        12.5  
3                     2             3        16.5  
4                     5             4         3.9  

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 62 entries, 0 to 61
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------